# E3 — Cross-Backbone Replication
## BERT × RoBERTa × DeBERTa — Calib-BERT v2 Reconstruction

This notebook tests whether the E2/E5 pattern generalizes beyond BERT.

### Primary question
Does the separation between **probability calibration** and **selective reliability** persist across Transformer backbones?

### Backbones
- **BERT-base-uncased** — frozen anchor from E2
- **RoBERTa-base** — fresh paired replication
- **DeBERTa-v3-base** — fresh paired replication

### Predeclared conditions per backbone
1. **Vanilla**
2. **Vanilla + Temperature Scaling**
3. **Label Smoothing ε = 0.10**

The LS value is transferred unchanged from the original BERT study.  
It is **not re-tuned per backbone**, which makes E3 a genuine cross-backbone replication rather than a new model-selection exercise.

### Training protocol
- Archived Dreaddit split assignment is reused unchanged.
- Classifier training uses **fit + validation** exactly as the final canonical classifier did.
- The 423-example calibration subset remains untouched during neural-network training.
- Temperature Scaling is fitted **only** on those 423 calibration examples.
- Official test set is evaluated only after the E3 design lock is written.
- Primary seed = **42**.
- This notebook does **not** select a “best” backbone from official-test performance.

### Why BERT is not retrained
The BERT fresh-paired run from E2 is already frozen. E3 reuses that prediction-level output as the BERT anchor and trains only RoBERTa and DeBERTa. This avoids unnecessary stochastic duplication and preserves the exact E2/E5 evidence chain.


In [ ]:
# ============================================================
# 0. USER CONFIGURATION
# ============================================================
from pathlib import Path

PROJECT_ROOT = Path("/content/calibbert_v2_e3")
INPUT_DIR = Path("/content")
WORK_DIR = PROJECT_ROOT / "work"
OUT_DIR = PROJECT_ROOT / "outputs_e3"
CACHE_DIR = PROJECT_ROOT / "hf_cache"

for p in [WORK_DIR, OUT_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PRIMARY_SEED = 42
MAX_LENGTH = 128
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
WARMUP_STEPS = 0

# Effective batch size is 16 for both new backbones.
BACKBONES = {
    "RoBERTa": {
        "model_id": "roberta-base",
        "micro_batch": 16,
        "grad_accum": 1,
    },
    "DeBERTa": {
        "model_id": "microsoft/deberta-v3-base",
        "micro_batch": 8,
        "grad_accum": 2,
    },
}

LS_EPSILON = 0.10

EXPECTED_TRAIN_N = 2838
EXPECTED_TEST_N = 715
EXPECTED_FIT_N = 1994
EXPECTED_VALIDATION_N = 421
EXPECTED_CALIBRATION_N = 423
EXPECTED_TRAIN_FINAL_N = EXPECTED_FIT_N + EXPECTED_VALIDATION_N

FIXED_COVERAGES = [0.90, 0.80, 0.70, 0.50]

EXPECTED_TRAIN_SHA256 = "4e791e60d3f04dd9f14abb1228b0d4f92bf628cc84097c1c173401e98d355f4e"
EXPECTED_TEST_SHA256 = "a3b6c9bf5d685f49d57e05d24266dddc52999843256b13e65c07267c543175fe"

print("Project:", PROJECT_ROOT)
print("Outputs:", OUT_DIR)


## 1. Colab preparation

Use a **T4 GPU** or better. E3 trains four new classifiers:

- RoBERTa Vanilla
- RoBERTa LS 0.10
- DeBERTa Vanilla
- DeBERTa LS 0.10

BERT is reused from E2 and is not retrained.

If `sentencepiece` is missing, the next cell installs only that small dependency required by DeBERTa-v3 tokenization.


In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("sentencepiece") is None:
    print("Installing sentencepiece...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"])
else:
    print("sentencepiece already available")


## 2. Imports and environment audit

In [ ]:
import os
import gc
import json
import math
import random
import hashlib
import zipfile
import platform
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import transformers
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

import sklearn
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    log_loss,
)

import scipy
from scipy.optimize import minimize_scalar

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

env = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "transformers": transformers.__version__,
    "scikit_learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
}

print(json.dumps(env, indent=2))
(OUT_DIR / "e3_environment.json").write_text(json.dumps(env, indent=2))

if not torch.cuda.is_available():
    raise RuntimeError("E3 requires a GPU runtime. In Colab select Runtime > Change runtime type > T4 GPU.")


## 3. Auto-discover E3 inputs

Upload these four lightweight/standard inputs directly into `/content/`:

1. `dreaddit-train.csv` or `dreaddit-train(1).csv`
2. `dreaddit-test.csv` or `dreaddit-test(1).csv`
3. `calibbert_e2_lite.zip`
4. `e2_prediction_level_long.csv`

Optional:
- `E2_CALIBRATION_LOCK.json`
- `e2_environment.json`


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def find_by_hash(directory, expected_hash, suffix=".csv"):
    for p in Path(directory).glob(f"*{suffix}"):
        try:
            if sha256_file(p) == expected_hash:
                return p
        except Exception:
            pass
    return None

TRAIN_PATH = find_by_hash(INPUT_DIR, EXPECTED_TRAIN_SHA256)
TEST_PATH = find_by_hash(INPUT_DIR, EXPECTED_TEST_SHA256)

LITE_ZIP = INPUT_DIR / "calibbert_e2_lite.zip"
E2_PRED_PATH = INPUT_DIR / "e2_prediction_level_long.csv"

if TRAIN_PATH is None:
    raise FileNotFoundError("Canonical Dreaddit train CSV not found in /content.")
if TEST_PATH is None:
    raise FileNotFoundError("Canonical Dreaddit test CSV not found in /content.")
if not LITE_ZIP.exists():
    raise FileNotFoundError("calibbert_e2_lite.zip not found in /content.")
if not E2_PRED_PATH.exists():
    raise FileNotFoundError("e2_prediction_level_long.csv not found in /content.")

print("Train :", TRAIN_PATH)
print("Test  :", TEST_PATH)
print("Lite  :", LITE_ZIP)
print("E2 pred:", E2_PRED_PATH)

print("\nSHA-256 verification")
print("Train:", sha256_file(TRAIN_PATH))
print("Test :", sha256_file(TEST_PATH))

assert sha256_file(TRAIN_PATH) == EXPECTED_TRAIN_SHA256
assert sha256_file(TEST_PATH) == EXPECTED_TEST_SHA256

print("Raw-data hashes: PASS")


## 4. Extract archived split and verify data alignment

In [ ]:
LITE_DIR = WORK_DIR / "e2_lite"
LITE_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(LITE_ZIP, "r") as z:
    z.extractall(LITE_DIR)

SPLIT_PATH = LITE_DIR / "development_split_assignment.csv"
if not SPLIT_PATH.exists():
    raise FileNotFoundError("development_split_assignment.csv missing from lite ZIP.")

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
split = pd.read_csv(SPLIT_PATH)

assert len(train_raw) == EXPECTED_TRAIN_N
assert len(test_raw) == EXPECTED_TEST_N

expected_split_counts = {
    "fit": EXPECTED_FIT_N,
    "validation": EXPECTED_VALIDATION_N,
    "calibration": EXPECTED_CALIBRATION_N,
}
actual_split_counts = split["split"].value_counts().to_dict()
print("Split counts:", actual_split_counts)

for k, v in expected_split_counts.items():
    assert actual_split_counts.get(k) == v, (k, actual_split_counts.get(k), v)

# Strong row-level verification against archived assignment.
check = train_raw.loc[
    split["row_index"].astype(int),
    ["id", "post_id", "label"]
].reset_index(drop=True)

assert np.array_equal(check["id"].astype(str).to_numpy(), split["id"].astype(str).to_numpy())
assert np.array_equal(check["post_id"].astype(str).to_numpy(), split["post_id"].astype(str).to_numpy())
assert np.array_equal(check["label"].astype(int).to_numpy(), split["label"].astype(int).to_numpy())

train_final_idx = split.loc[
    split["split"].isin(["fit", "validation"]), "row_index"
].astype(int).to_numpy()

calibration_idx = split.loc[
    split["split"] == "calibration", "row_index"
].astype(int).to_numpy()

train_final_df = train_raw.iloc[train_final_idx].reset_index(drop=True)
calibration_df = train_raw.iloc[calibration_idx].reset_index(drop=True)
test_df = test_raw.reset_index(drop=True)

assert len(train_final_df) == EXPECTED_TRAIN_FINAL_N
assert len(calibration_df) == EXPECTED_CALIBRATION_N
assert len(test_df) == EXPECTED_TEST_N

print("Archived split alignment: PASS")
print("train_final:", len(train_final_df))
print("calibration:", len(calibration_df))
print("official test:", len(test_df))


## 5. Load frozen BERT anchor from E2

Only three E2 BERT conditions are imported:
- Vanilla
- Vanilla + Temperature Scaling
- LS 0.10

This is the frozen E2 evidence chain used in E5.


In [ ]:
e2_pred = pd.read_csv(E2_PRED_PATH)

required_e2 = {
    "row_index", "id", "post_id", "true_label",
    "model", "calibrator", "p_not_stress", "p_stress",
    "predicted_label", "confidence"
}
missing = required_e2 - set(e2_pred.columns)
if missing:
    raise KeyError(f"E2 prediction file missing columns: {sorted(missing)}")

bert_map = [
    ("M0_Vanilla_BERT", "none", "Vanilla"),
    ("M0_Vanilla_BERT", "temperature", "Vanilla+TS"),
    ("M1_LS_0.10", "none", "LS0.10"),
]

bert_anchor_parts = []
for model, cal, condition in bert_map:
    x = e2_pred[
        (e2_pred["model"] == model) &
        (e2_pred["calibrator"] == cal)
    ].copy().sort_values("row_index")

    if len(x) != EXPECTED_TEST_N:
        raise RuntimeError(f"BERT E2 anchor incomplete for {model}/{cal}: {len(x)}")

    # Verify test identity/order against raw official test.
    assert np.array_equal(x["id"].astype(str).to_numpy(), test_df["id"].astype(str).to_numpy())
    assert np.array_equal(x["post_id"].astype(str).to_numpy(), test_df["post_id"].astype(str).to_numpy())
    assert np.array_equal(x["true_label"].astype(int).to_numpy(), test_df["label"].astype(int).to_numpy())

    x["backbone"] = "BERT"
    x["condition"] = condition
    bert_anchor_parts.append(x)

bert_anchor = pd.concat(bert_anchor_parts, ignore_index=True)
print("Frozen BERT anchor rows:", len(bert_anchor))
print(bert_anchor.groupby(["backbone", "condition"]).size())


## 6. Resolve and lock exact Hugging Face model revisions

This cell resolves the current repository commit for each new backbone **before training**.  
Subsequent tokenizer/model downloads are pinned to that resolved commit hash for reproducibility.


In [ ]:
revision_lock = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "primary_seed": PRIMARY_SEED,
    "backbones": {},
}

for backbone, spec in BACKBONES.items():
    cfg = AutoConfig.from_pretrained(
        spec["model_id"],
        cache_dir=CACHE_DIR,
    )
    commit = getattr(cfg, "_commit_hash", None)
    if not commit:
        raise RuntimeError(
            f"Could not resolve immutable commit hash for {spec['model_id']}."
        )

    revision_lock["backbones"][backbone] = {
        "model_id": spec["model_id"],
        "revision_commit": commit,
        "micro_batch": spec["micro_batch"],
        "gradient_accumulation": spec["grad_accum"],
        "effective_batch": spec["micro_batch"] * spec["grad_accum"],
    }

(OUT_DIR / "E3_MODEL_REVISION_LOCK.json").write_text(
    json.dumps(revision_lock, indent=2)
)

print(json.dumps(revision_lock, indent=2))


## 7. Write E3 design lock **before training**

The lock fixes:
- backbones,
- seed,
- training hyperparameters,
- LS ε,
- calibration partition,
- primary conditions,
- official-test metrics.

E3 validity does not depend on any method becoming “best”.


In [ ]:
e3_lock = {
    "analysis": "E3 Cross-Backbone Replication",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "canonical_train_sha256": EXPECTED_TRAIN_SHA256,
    "canonical_test_sha256": EXPECTED_TEST_SHA256,
    "split_source": "archived development_split_assignment.csv",
    "bert_anchor_source": "frozen E2 e2_prediction_level_long.csv",
    "new_backbones": revision_lock["backbones"],
    "primary_seed": PRIMARY_SEED,
    "training": {
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "max_length": MAX_LENGTH,
        "warmup_steps": WARMUP_STEPS,
        "grad_clip": GRAD_CLIP,
        "train_partition": "fit + validation (n=2415)",
        "calibration_partition": "calibration only (n=423)",
    },
    "conditions_per_backbone": [
        "Vanilla",
        "Vanilla + Temperature Scaling",
        "Label Smoothing epsilon=0.10",
    ],
    "label_smoothing_epsilon": LS_EPSILON,
    "ls_retuned_per_backbone": False,
    "temperature_fit_partition": "calibration only (n=423)",
    "primary_probability_endpoints": ["brier", "nll"],
    "secondary_calibration_endpoints": ["ece", "ace", "hce_0.90"],
    "selective_endpoint": "AURC",
    "fixed_coverages": FIXED_COVERAGES,
    "official_test_used_for_training_or_calibrator_fit": False,
    "test_based_backbone_selection": False,
}

(OUT_DIR / "E3_ANALYSIS_LOCK.json").write_text(json.dumps(e3_lock, indent=2))
print(json.dumps(e3_lock, indent=2))


## 8. Determinism, dataset, and model helpers

In [ ]:
def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

class TextDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=128):
        texts = frame["text"].fillna("").astype(str).tolist()
        self.enc = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        self.labels = torch.tensor(frame["label"].astype(int).to_numpy(), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        item["labels"] = self.labels[idx]
        return item

def make_loader(ds, batch_size, shuffle, seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=g,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

def soft_target_cross_entropy(logits, labels, epsilon):
    K = logits.shape[-1]
    target = torch.full(
        (labels.shape[0], K),
        epsilon / (K - 1),
        dtype=torch.float32,
        device=labels.device,
    )
    target.scatter_(1, labels.unsqueeze(1), 1.0 - epsilon)
    return -(target * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

@torch.no_grad()
def predict_logits(model, dataset, batch_size):
    loader = make_loader(dataset, batch_size, False, PRIMARY_SEED)
    model.eval()

    logits_all = []
    labels_all = []

    for batch in loader:
        labels = batch.pop("labels")
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        outputs = model(**batch)
        logits_all.append(outputs.logits.detach().cpu().numpy())
        labels_all.append(labels.numpy())

    return np.concatenate(logits_all), np.concatenate(labels_all)


## 9. Train one fresh backbone condition with checkpoint-like logits caching

Each completed neural-network condition saves:
- calibration logits
- official-test logits
- labels
- epoch training history

If the same `.npz` output already exists in the current runtime, the notebook reuses it instead of retraining that condition.


In [ ]:
def train_condition(backbone, mode):
    spec = BACKBONES[backbone]
    model_id = spec["model_id"]
    revision = revision_lock["backbones"][backbone]["revision_commit"]
    micro_batch = spec["micro_batch"]
    grad_accum = spec["grad_accum"]

    condition_name = "Vanilla" if mode == "none" else "LS0.10"
    cache_npz = OUT_DIR / f"{backbone}_{condition_name}_seed{PRIMARY_SEED}_logits.npz"
    history_csv = OUT_DIR / f"{backbone}_{condition_name}_seed{PRIMARY_SEED}_history.csv"

    if cache_npz.exists():
        print(f"[REUSE] {cache_npz.name}")
        arr = np.load(cache_npz)
        history = pd.read_csv(history_csv) if history_csv.exists() else pd.DataFrame()
        return {
            "cal_logits": arr["cal_logits"],
            "cal_y": arr["cal_y"],
            "test_logits": arr["test_logits"],
            "test_y": arr["test_y"],
            "history": history,
            "revision": revision,
        }

    print(f"\n===== TRAIN {backbone} / {condition_name} =====")
    print("model_id:", model_id)
    print("revision:", revision)
    print("micro_batch:", micro_batch, "grad_accum:", grad_accum)

    set_all_seeds(PRIMARY_SEED)

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        revision=revision,
        cache_dir=CACHE_DIR,
        use_fast=True,
    )

    train_ds = TextDataset(train_final_df, tokenizer, MAX_LENGTH)
    cal_ds = TextDataset(calibration_df, tokenizer, MAX_LENGTH)
    test_ds = TextDataset(test_df, tokenizer, MAX_LENGTH)

    train_loader = make_loader(
        train_ds, micro_batch, True, PRIMARY_SEED
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        revision=revision,
        cache_dir=CACHE_DIR,
        num_labels=2,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    optimizer_steps_per_epoch = math.ceil(len(train_loader) / grad_accum)
    total_optimizer_steps = optimizer_steps_per_epoch * EPOCHS

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=WARMUP_STEPS,
        num_training_steps=total_optimizer_steps,
    )

    history_rows = []
    optimizer.zero_grad(set_to_none=True)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = []
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(train_loader, start=1):
            labels = batch.pop("labels").to(DEVICE)
            inputs = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

            outputs = model(**inputs)

            if mode == "none":
                raw_loss = F.cross_entropy(outputs.logits, labels)
            elif mode == "ls":
                raw_loss = soft_target_cross_entropy(
                    outputs.logits, labels, LS_EPSILON
                )
            else:
                raise ValueError(mode)

            loss = raw_loss / grad_accum
            loss.backward()
            running.append(float(raw_loss.detach().cpu()))

            should_step = (step % grad_accum == 0) or (step == len(train_loader))
            if should_step:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        mean_loss = float(np.mean(running))
        history_rows.append({
            "backbone": backbone,
            "condition": condition_name,
            "seed": PRIMARY_SEED,
            "epoch": epoch,
            "train_loss": mean_loss,
        })
        print(f"epoch {epoch}/{EPOCHS} mean train loss = {mean_loss:.6f}")

    cal_logits, cal_y = predict_logits(model, cal_ds, max(8, micro_batch))
    test_logits, test_y = predict_logits(model, test_ds, max(8, micro_batch))

    np.savez_compressed(
        cache_npz,
        cal_logits=cal_logits,
        cal_y=cal_y,
        test_logits=test_logits,
        test_y=test_y,
    )

    history = pd.DataFrame(history_rows)
    history.to_csv(history_csv, index=False)

    del model, tokenizer, train_ds, cal_ds, test_ds, train_loader
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "cal_logits": cal_logits,
        "cal_y": cal_y,
        "test_logits": test_logits,
        "test_y": test_y,
        "history": history,
        "revision": revision,
    }


## 10. Run the four new neural-network trainings

Expected on a T4:
- RoBERTa first
- DeBERTa second

Do not interrupt an individual training condition.  
After each condition finishes, its logits are saved immediately.


In [ ]:
fresh_runs = {}
all_histories = []

for backbone in ["RoBERTa", "DeBERTa"]:
    for mode in ["none", "ls"]:
        key = (backbone, mode)
        result = train_condition(backbone, mode)
        fresh_runs[key] = result
        if len(result["history"]):
            all_histories.append(result["history"])

        # Strong label check against canonical partitions.
        assert np.array_equal(
            result["cal_y"],
            calibration_df["label"].astype(int).to_numpy()
        )
        assert np.array_equal(
            result["test_y"],
            test_df["label"].astype(int).to_numpy()
        )

if all_histories:
    pd.concat(all_histories, ignore_index=True).to_csv(
        OUT_DIR / "e3_training_history.csv", index=False
    )

print("All fresh backbone trainings complete.")


## 11. Temperature Scaling fitted on calibration only

In [ ]:
def softmax_np(logits):
    x = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

def nll_from_logits(logits, y):
    probs = softmax_np(logits)
    return float(log_loss(y, probs, labels=[0, 1]))

def fit_temperature(cal_logits, y):
    def objective(logT):
        T = float(np.exp(logT))
        return nll_from_logits(cal_logits / T, y)

    result = minimize_scalar(
        objective,
        bounds=(-4.0, 4.0),
        method="bounded",
        options={"xatol": 1e-10},
    )

    T = float(np.exp(result.x))
    return {
        "temperature": T,
        "calibration_nll_before": nll_from_logits(cal_logits, y),
        "calibration_nll_after": nll_from_logits(cal_logits / T, y),
        "optimizer_success": bool(result.success),
    }

ts_rows = []
temperatures = {}

for backbone in ["RoBERTa", "DeBERTa"]:
    r = fresh_runs[(backbone, "none")]
    fit = fit_temperature(r["cal_logits"], r["cal_y"])
    temperatures[backbone] = fit["temperature"]

    ts_rows.append({
        "backbone": backbone,
        **fit
    })

ts_table = pd.DataFrame(ts_rows)
ts_table.to_csv(OUT_DIR / "e3_temperature_scaling_parameters.csv", index=False)
display(ts_table)


## 12. Metrics and selective-reliability functions

In [ ]:
def ece_equal_width(y_true, probs, n_bins=10):
    pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    correct = (pred == y_true).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for b in range(n_bins):
        lo, hi = edges[b], edges[b + 1]
        mask = (conf >= lo) & (
            (conf <= hi) if b == n_bins - 1 else (conf < hi)
        )
        if mask.any():
            ece += mask.mean() * abs(
                correct[mask].mean() - conf[mask].mean()
            )
    return float(ece)

def ace_equal_frequency(y_true, probs, n_bins=10):
    pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    correct = (pred == y_true).astype(float)
    order = np.argsort(conf)

    ace = 0.0
    for idx in np.array_split(order, n_bins):
        if len(idx):
            ace += (len(idx) / len(y_true)) * abs(
                correct[idx].mean() - conf[idx].mean()
            )
    return float(ace)

def aurc_and_fixed_coverage(y_true, pred, conf):
    order = np.lexsort((np.arange(len(conf)), -conf))
    y_s = y_true[order]
    p_s = pred[order]

    errors = (p_s != y_s).astype(int)
    k = np.arange(1, len(y_true) + 1)
    risk = np.cumsum(errors) / k
    coverage = k / len(y_true)
    aurc = float(np.mean(risk))

    true_stress = (y_s == 1)
    pred_stress = (p_s == 1)
    total_true_stress = int((y_true == 1).sum())
    tp_retained = np.cumsum((true_stress & pred_stress).astype(int))
    stress_recall = tp_retained / total_true_stress

    fixed = {}
    for target in FIXED_COVERAGES:
        kk = max(1, min(len(y_true), int(round(target * len(y_true)))))
        fixed[target] = {
            "actual_coverage": float(coverage[kk - 1]),
            "risk": float(risk[kk - 1]),
            "stress_recall": float(stress_recall[kk - 1]),
            "confidence_cutoff": float(conf[order][kk - 1]),
        }

    return aurc, fixed, order, risk, coverage, stress_recall

def metric_bundle(y_true, probs):
    pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred, average="binary", pos_label=1, zero_division=0
    )

    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    aurc, fixed, order, risk_curve, coverage_curve, stress_curve = (
        aurc_and_fixed_coverage(y_true, pred, conf)
    )

    out = {
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "brier": float(np.mean((probs[:, 1] - y_true) ** 2)),
        "nll": float(log_loss(y_true, probs, labels=[0, 1])),
        "ece": ece_equal_width(y_true, probs),
        "ace": ace_equal_frequency(y_true, probs),
        "hce_0.90": int(((pred != y_true) & (conf >= 0.90)).sum()),
        "coverage_0.90": float((conf >= 0.90).mean()),
        "selective_error_0.90": (
            float((pred[conf >= 0.90] != y_true[conf >= 0.90]).mean())
            if (conf >= 0.90).any() else np.nan
        ),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
        "aurc": aurc,
    }

    for cov, d in fixed.items():
        pct = int(round(cov * 100))
        out[f"risk_at_{pct}pct_coverage"] = d["risk"]
        out[f"stress_recall_at_{pct}pct_coverage"] = d["stress_recall"]
        out[f"confidence_cutoff_at_{pct}pct_coverage"] = d["confidence_cutoff"]

    return out


## 13. Build RoBERTa and DeBERTa prediction-level outputs

Primary E3 conditions:
- Vanilla
- Vanilla+TS
- LS0.10

Temperature Scaling is applied only to the corresponding Vanilla logits.


In [ ]:
new_prediction_parts = []

for backbone in ["RoBERTa", "DeBERTa"]:
    # Vanilla
    r0 = fresh_runs[(backbone, "none")]
    y = r0["test_y"]
    logits = r0["test_logits"]

    for condition, transformed_logits in [
        ("Vanilla", logits),
        ("Vanilla+TS", logits / temperatures[backbone]),
    ]:
        probs = softmax_np(transformed_logits)
        pred_y = probs.argmax(axis=1)
        conf = probs.max(axis=1)

        x = pd.DataFrame({
            "row_index": np.arange(EXPECTED_TEST_N),
            "id": test_df["id"].to_numpy(),
            "post_id": test_df["post_id"].to_numpy(),
            "true_label": y,
            "backbone": backbone,
            "condition": condition,
            "p_not_stress": probs[:, 0],
            "p_stress": probs[:, 1],
            "predicted_label": pred_y,
            "confidence": conf,
        })
        new_prediction_parts.append(x)

    # LS
    r1 = fresh_runs[(backbone, "ls")]
    probs = softmax_np(r1["test_logits"])
    x = pd.DataFrame({
        "row_index": np.arange(EXPECTED_TEST_N),
        "id": test_df["id"].to_numpy(),
        "post_id": test_df["post_id"].to_numpy(),
        "true_label": r1["test_y"],
        "backbone": backbone,
        "condition": "LS0.10",
        "p_not_stress": probs[:, 0],
        "p_stress": probs[:, 1],
        "predicted_label": probs.argmax(axis=1),
        "confidence": probs.max(axis=1),
    })
    new_prediction_parts.append(x)

new_predictions = pd.concat(new_prediction_parts, ignore_index=True)
print(new_predictions.groupby(["backbone", "condition"]).size())


## 14. Normalize BERT anchor into the E3 schema

In [ ]:
bert_e3_parts = []

for model, cal, condition in bert_map:
    x = e2_pred[
        (e2_pred["model"] == model) &
        (e2_pred["calibrator"] == cal)
    ].copy().sort_values("row_index")

    y = x[[
        "row_index", "id", "post_id", "true_label",
        "p_not_stress", "p_stress",
        "predicted_label", "confidence"
    ]].copy()

    y["backbone"] = "BERT"
    y["condition"] = condition

    bert_e3_parts.append(y)

bert_e3 = pd.concat(bert_e3_parts, ignore_index=True)

all_predictions = pd.concat(
    [bert_e3, new_predictions],
    ignore_index=True,
    sort=False
)

all_predictions = all_predictions[[
    "row_index", "id", "post_id", "true_label",
    "backbone", "condition",
    "p_not_stress", "p_stress",
    "predicted_label", "confidence"
]]

assert all_predictions.groupby(["backbone", "condition"])["row_index"].nunique().eq(EXPECTED_TEST_N).all()

all_predictions.to_csv(
    OUT_DIR / "e3_prediction_level_long.csv",
    index=False
)

display(
    all_predictions.groupby(["backbone", "condition"])
    .size()
    .rename("n")
    .reset_index()
)


## 15. Cross-backbone official-test metrics

In [ ]:
metrics_rows = []

for (backbone, condition), x in all_predictions.groupby(["backbone", "condition"]):
    x = x.sort_values("row_index")
    probs = x[["p_not_stress", "p_stress"]].to_numpy(float)
    y = x["true_label"].to_numpy(int)

    row = {
        "backbone": backbone,
        "condition": condition,
        **metric_bundle(y, probs),
    }

    if condition == "Vanilla+TS":
        if backbone in temperatures:
            row["temperature"] = temperatures[backbone]
        else:
            row["temperature"] = np.nan
    else:
        row["temperature"] = np.nan

    metrics_rows.append(row)

metrics = pd.DataFrame(metrics_rows)

condition_order = {"Vanilla": 0, "Vanilla+TS": 1, "LS0.10": 2}
backbone_order = {"BERT": 0, "RoBERTa": 1, "DeBERTa": 2}

metrics["_b"] = metrics["backbone"].map(backbone_order)
metrics["_c"] = metrics["condition"].map(condition_order)
metrics = metrics.sort_values(["_b", "_c"]).drop(columns=["_b", "_c"]).reset_index(drop=True)

metrics.to_csv(OUT_DIR / "e3_official_test_metrics.csv", index=False)

display(metrics[[
    "backbone", "condition",
    "accuracy", "f1",
    "brier", "nll", "ece", "ace",
    "hce_0.90", "coverage_0.90", "aurc"
]])


## 16. Temperature Scaling decision/rank invariance audit

For each backbone, a positive scalar Temperature Scaling transformation should:
- preserve predicted class labels,
- preserve confidence ranking,
- preserve rank-based AURC.

The absolute confidence scale can still change substantially.


In [ ]:
invariance_rows = []

for backbone in ["BERT", "RoBERTa", "DeBERTa"]:
    a = all_predictions[
        (all_predictions["backbone"] == backbone) &
        (all_predictions["condition"] == "Vanilla")
    ].sort_values("row_index")

    b = all_predictions[
        (all_predictions["backbone"] == backbone) &
        (all_predictions["condition"] == "Vanilla+TS")
    ].sort_values("row_index")

    same_predictions = bool(np.array_equal(
        a["predicted_label"].to_numpy(),
        b["predicted_label"].to_numpy()
    ))

    rank_a = a.sort_values(
        ["confidence", "row_index"],
        ascending=[False, True],
        kind="mergesort"
    )["row_index"].to_numpy()

    rank_b = b.sort_values(
        ["confidence", "row_index"],
        ascending=[False, True],
        kind="mergesort"
    )["row_index"].to_numpy()

    same_ranking = bool(np.array_equal(rank_a, rank_b))

    aurc_a = float(metrics.loc[
        (metrics["backbone"] == backbone) &
        (metrics["condition"] == "Vanilla"),
        "aurc"
    ].iloc[0])

    aurc_b = float(metrics.loc[
        (metrics["backbone"] == backbone) &
        (metrics["condition"] == "Vanilla+TS"),
        "aurc"
    ].iloc[0])

    invariance_rows.append({
        "backbone": backbone,
        "same_predicted_labels": same_predictions,
        "same_confidence_ranking": same_ranking,
        "aurc_vanilla": aurc_a,
        "aurc_temperature": aurc_b,
        "absolute_aurc_difference": abs(aurc_a - aurc_b),
    })

invariance = pd.DataFrame(invariance_rows)
invariance.to_csv(
    OUT_DIR / "e3_temperature_rank_invariance.csv",
    index=False
)
display(invariance)


## 17. Directional replication effects

This table is central to E3.

For each backbone we compute:

### Temperature Scaling effect relative to Vanilla
- Δ Brier
- Δ NLL
- Δ ECE
- Δ AURC
- Δ F1

### LS 0.10 effect relative to Vanilla
- the same set of deltas

Negative Δ for Brier/NLL/ECE/AURC means the endpoint decreased.  
No external “winner” is assigned; E3 records direction and magnitude.


In [ ]:
effect_rows = []

for backbone in ["BERT", "RoBERTa", "DeBERTa"]:
    base = metrics[
        (metrics["backbone"] == backbone) &
        (metrics["condition"] == "Vanilla")
    ].iloc[0]

    for condition in ["Vanilla+TS", "LS0.10"]:
        comp = metrics[
            (metrics["backbone"] == backbone) &
            (metrics["condition"] == condition)
        ].iloc[0]

        effect_rows.append({
            "backbone": backbone,
            "comparison": f"{condition} minus Vanilla",
            "delta_accuracy": comp["accuracy"] - base["accuracy"],
            "delta_f1": comp["f1"] - base["f1"],
            "delta_brier": comp["brier"] - base["brier"],
            "delta_nll": comp["nll"] - base["nll"],
            "delta_ece": comp["ece"] - base["ece"],
            "delta_ace": comp["ace"] - base["ace"],
            "delta_hce_0.90": comp["hce_0.90"] - base["hce_0.90"],
            "delta_aurc": comp["aurc"] - base["aurc"],
        })

effects = pd.DataFrame(effect_rows)
effects.to_csv(OUT_DIR / "e3_directional_effects.csv", index=False)
display(effects)


## 18. Publication-ready E3 table

One row per backbone × condition with:
- discrimination,
- probability calibration,
- high-confidence error,
- selective prediction.


In [ ]:
publication_cols = [
    "backbone", "condition",
    "accuracy", "precision", "recall", "f1",
    "brier", "nll", "ece", "ace",
    "hce_0.90", "coverage_0.90",
    "selective_error_0.90",
    "aurc",
    "risk_at_90pct_coverage",
    "risk_at_80pct_coverage",
    "risk_at_70pct_coverage",
    "risk_at_50pct_coverage",
    "stress_recall_at_90pct_coverage",
    "stress_recall_at_80pct_coverage",
    "stress_recall_at_70pct_coverage",
    "stress_recall_at_50pct_coverage",
]

table_e3 = metrics[publication_cols].copy()
table_e3.to_csv(
    OUT_DIR / "Table_E3_Cross_Backbone_Replication.csv",
    index=False
)

display(table_e3)


## 19. Figures

In [ ]:
import matplotlib.pyplot as plt

# Figure 1 — Brier
fig, ax = plt.subplots(figsize=(8, 5.5))
for condition in ["Vanilla", "Vanilla+TS", "LS0.10"]:
    x = metrics[metrics["condition"] == condition]
    ax.plot(x["backbone"], x["brier"], marker="o", label=condition)
ax.set_xlabel("Backbone")
ax.set_ylabel("Brier score")
ax.set_title("Cross-Backbone Probability Calibration — Brier")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "Figure_E3_Brier.png", dpi=300, bbox_inches="tight")
fig.savefig(OUT_DIR / "Figure_E3_Brier.pdf", bbox_inches="tight")
plt.show()

# Figure 2 — ECE
fig, ax = plt.subplots(figsize=(8, 5.5))
for condition in ["Vanilla", "Vanilla+TS", "LS0.10"]:
    x = metrics[metrics["condition"] == condition]
    ax.plot(x["backbone"], x["ece"], marker="o", label=condition)
ax.set_xlabel("Backbone")
ax.set_ylabel("ECE")
ax.set_title("Cross-Backbone Confidence Calibration — ECE")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "Figure_E3_ECE.png", dpi=300, bbox_inches="tight")
fig.savefig(OUT_DIR / "Figure_E3_ECE.pdf", bbox_inches="tight")
plt.show()

# Figure 3 — AURC
fig, ax = plt.subplots(figsize=(8, 5.5))
for condition in ["Vanilla", "Vanilla+TS", "LS0.10"]:
    x = metrics[metrics["condition"] == condition]
    ax.plot(x["backbone"], x["aurc"], marker="o", label=condition)
ax.set_xlabel("Backbone")
ax.set_ylabel("AURC")
ax.set_title("Cross-Backbone Selective Reliability — AURC")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "Figure_E3_AURC.png", dpi=300, bbox_inches="tight")
fig.savefig(OUT_DIR / "Figure_E3_AURC.pdf", bbox_inches="tight")
plt.show()


## 20. Package integrity and downloadable ZIP

This cell:
- hashes all current E3 outputs,
- writes a manifest,
- creates a single ZIP for archival/download.

Run it after all preceding cells finish.


In [ ]:
def hash_outputs(directory):
    rows = []
    for p in sorted(Path(directory).iterdir()):
        if not p.is_file():
            continue
        if p.name in {"E3_OUTPUT_HASHES.csv", "E3_MANIFEST.json", "E3_RESULTS.zip"}:
            continue
        rows.append({
            "filename": p.name,
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        })
    return pd.DataFrame(rows)

hashes = hash_outputs(OUT_DIR)
hashes.to_csv(OUT_DIR / "E3_OUTPUT_HASHES.csv", index=False)

manifest = {
    "package": "CalibBERT_v2_E3_Cross_Backbone_Replication",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "canonical_train_sha256": EXPECTED_TRAIN_SHA256,
    "canonical_test_sha256": EXPECTED_TEST_SHA256,
    "primary_seed": PRIMARY_SEED,
    "backbones": ["BERT", "RoBERTa", "DeBERTa"],
    "conditions": ["Vanilla", "Vanilla+TS", "LS0.10"],
    "files": hashes.to_dict(orient="records"),
}

(OUT_DIR / "E3_MANIFEST.json").write_text(
    json.dumps(manifest, indent=2)
)

ZIP_PATH = PROJECT_ROOT / "E3_RESULTS.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT_DIR.iterdir()):
        if p.is_file():
            z.write(p, arcname=p.name)

print("E3 archive:", ZIP_PATH)
print("Files:", len(list(OUT_DIR.iterdir())))
print("Download E3_RESULTS.zip from the Colab Files panel.")


## 21. E3 completion gate

E3 can be frozen when:

- [ ] canonical raw-data SHA-256 hashes pass
- [ ] archived 1,994 / 421 / 423 split is reused unchanged
- [ ] BERT anchor comes from frozen E2 predictions
- [ ] RoBERTa and DeBERTa model revisions are immutable commit hashes
- [ ] E3 design lock is written before fresh model training
- [ ] RoBERTa Vanilla and LS0.10 complete
- [ ] DeBERTa Vanilla and LS0.10 complete
- [ ] Temperature Scaling is fitted only on calibration n=423
- [ ] official test has exactly 715 aligned examples for all 9 primary conditions
- [ ] TS label/rank invariance is audited for all three backbones
- [ ] cross-backbone Brier/NLL/ECE/AURC effects are exported
- [ ] no official-test-based backbone or method selection is performed
- [ ] `E3_RESULTS.zip` is downloaded and archived

### Main files to send back for audit
1. `Table_E3_Cross_Backbone_Replication.csv`
2. `e3_directional_effects.csv`
3. `e3_temperature_rank_invariance.csv`
4. `e3_temperature_scaling_parameters.csv`
5. `E3_ANALYSIS_LOCK.json`
6. `E3_MODEL_REVISION_LOCK.json`
7. `E3_RESULTS.zip`

### After E3
If cross-backbone evidence is stable, proceed to **E4 distribution-shift audit**.  
If effects differ materially by backbone, that heterogeneity itself becomes part of the revised scientific claim and should be quantified before E4.
